# Field Validation — Domain-Shift Pilot

Scores the trained classifier on real Sunly Solar field photos, compared
against a trivial brightness-threshold baseline. This is the paper's key
empirical contribution — Kaggle-trained model tested on independently-
sourced field photos.

Explicitly a **preliminary domain-shift pilot**, not a claim of
generalization (see project brief). A small n (target 15-30 photos) means
we can report accuracy on this set, but not confidence intervals or
statistical generalization.

**BUG FIX (this version):** the previous version of this notebook referenced
`mean_brightness()`, `best_direction`, and `best_thresh` in the scoring cell
without ever defining them — it would have crashed. Section 3 below builds
the missing brightness-baseline calibration step properly, on the Kaggle
val set (not field photos, to keep the domain-shift comparison fair).

## 1. Setup

In [1]:
!pip install ultralytics -q
from ultralytics import YOLO
import numpy as np
import pandas as pd
from PIL import Image
import os, glob, re

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


## 2. Load the trained classifier

Upload `best.pt` when prompted.

In [2]:
from google.colab import files
print("Upload best.pt")
uploaded = files.upload()
model = YOLO(list(uploaded.keys())[0])

Upload best.pt


Saving best.pt to best.pt


## 3. Download Kaggle dataset (for baseline calibration)

Same Secrets-based credential pattern as the other notebooks.

In [3]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!kaggle datasets download -d hemanthsai7/solar-panel-dust-detection -q
!unzip -q solar-panel-dust-detection.zip -d /content/data

Dataset URL: https://www.kaggle.com/datasets/hemanthsai7/solar-panel-dust-detection
License(s): CC0-1.0


## 4. Calibrate the brightness-threshold baseline

A trivial non-ML baseline: classify "dusty" based purely on average image
brightness. Calibrated on the **same val split (seed=42) used to evaluate
the real classifier** — not on field photos — so the domain-shift
comparison is fair (both methods are tested on genuinely unseen field
data, neither is tuned on it).

Scans a range of brightness thresholds and both possible directions
(dusty panels could plausibly be darker OR brighter than clean ones,
depending on lighting/dust color — don't assume, check both), picks
whichever combination maximizes accuracy on the Kaggle val set.

In [4]:
def mean_brightness(path):
    """Average grayscale pixel value, 0-255."""
    img = Image.open(path).convert('L')
    return float(np.array(img).mean())


# Reproduce the IDENTICAL val split used in baseline_classifier_training.ipynb
import random
random.seed(42)
SOURCE_DIR = '/content/data/Detect_solar_dust'
CLASSES = ['Clean', 'Dusty']
VAL_FRACTION = 0.2

val_records = []
for cls in CLASSES:
    src_folder = os.path.join(SOURCE_DIR, cls)
    images = os.listdir(src_folder)
    random.shuffle(images)
    n_val = int(len(images) * VAL_FRACTION)
    val_images = images[:n_val]
    for img in val_images:
        path = os.path.join(src_folder, img)
        val_records.append({'path': path, 'true_label': cls, 'brightness': mean_brightness(path)})

val_df = pd.DataFrame(val_records)
print(f"Calibrating on {len(val_df)} val images (same split as classifier training)")

# Scan thresholds x both directions, pick the best on this val set
thresholds = np.arange(val_df['brightness'].min(), val_df['brightness'].max(), 1.0)
best_thresh, best_direction, best_acc = None, None, 0.0

for t in thresholds:
    for direction in ['dusty_below', 'dusty_above']:
        if direction == 'dusty_below':
            preds = np.where(val_df['brightness'] < t, 'Dusty', 'Clean')
        else:
            preds = np.where(val_df['brightness'] > t, 'Dusty', 'Clean')
        acc = (preds == val_df['true_label']).mean()
        if acc > best_acc:
            best_acc, best_thresh, best_direction = acc, t, direction

print(f"Best brightness baseline: threshold={best_thresh:.1f}, direction={best_direction}, val accuracy={best_acc:.3f}")
print("This threshold/direction is now FROZEN and will be applied as-is to field photos below - not re-tuned on them.")

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Calibrating on 511 val images (same split as classifier training)
Best brightness baseline: threshold=45.0, direction=dusty_below, val accuracy=0.585
This threshold/direction is now FROZEN and will be applied as-is to field photos below - not re-tuned on them.


## 5. Upload field photos

Expected filename format: `system_p{panel}_{YYYYMMDD}_{clean|dusty}.jpg`
(e.g. `Ali_p1_20260901_clean.jpg`) — matches the capture protocol. Accepts
either a single zip of photos or individual file uploads. Broadened to
accept .jpg/.jpeg/.png (previous version only matched .jpg).

In [5]:
import zipfile
import os
import glob
from google.colab import files

# Upload field photos ZIP
uploaded_field = files.upload()

# Clear previous field photos
!rm -rf /content/field_photos
os.makedirs('/content/field_photos', exist_ok=True)

# Get uploaded filename
first_name = list(uploaded_field.keys())[0]

# Extract ZIP or copy individual files
if first_name.lower().endswith('.zip'):
    with zipfile.ZipFile(first_name, 'r') as zf:
        zf.extractall('/content/field_photos')
else:
    for fname in uploaded_field:
        os.rename(fname, f'/content/field_photos/{fname}')

# Find all images
field_paths = []

for ext in ('jpg', 'jpeg', 'png', 'JPG', 'JPEG', 'PNG'):
    field_paths += glob.glob(
        f'/content/field_photos/**/*.{ext}',
        recursive=True
    )

print(f"{len(field_paths)} field photos loaded")

Saving Field Photos.zip to Field Photos (1).zip
31 field photos loaded


## 6. Parse filenames into system / panel / date / true label

In [9]:
def parse_filename(path):
    fname = os.path.basename(path)

    m = re.match(
        r'([a-zA-Z]+)p?(\d*)_?(\d{8})_(clean|dusty)',
        fname,
        re.IGNORECASE
    )

    if not m:
        print(f"WARNING: couldn't parse {fname}, skipping")
        return None

    system, panel, date, label = m.groups()

    return {
        'path': path,
        'system': system.lower(),
        'panel': panel or '1',
        'date': date,
        'label': label.capitalize()
    }

field_records = [
    r for r in (parse_filename(p) for p in field_paths)
    if r
]

print(f"{len(field_records)} photos parsed successfully")

31 photos parsed successfully


## 7. Run both methods on field photos

Real classifier: straight domain-shift inference, no retraining. Brightness
baseline: uses the threshold/direction frozen in Section 4.

In [10]:
results = []
for rec in field_records:
    # Classifier (no retraining, straight domain-shift inference)
    pred = model.predict(rec['path'], verbose=False)[0]
    cls_label = pred.names[pred.probs.top1]
    cls_conf = float(pred.probs.top1conf)

    # Frozen brightness baseline
    b = mean_brightness(rec['path'])
    if best_direction == 'dusty_below':
        baseline_label = 'Dusty' if b < best_thresh else 'Clean'
    else:
        baseline_label = 'Dusty' if b > best_thresh else 'Clean'

    results.append({**rec, 'classifier_pred': cls_label, 'classifier_conf': cls_conf,
                     'baseline_pred': baseline_label, 'brightness': b})

## 8. Score both methods

In [11]:
df = pd.DataFrame(results)
df['classifier_correct'] = df['classifier_pred'] == df['label']
df['baseline_correct'] = df['baseline_pred'] == df['label']

print(f"Field photos (n={len(df)})")
print(f"Classifier accuracy: {df['classifier_correct'].mean():.3f}")
print(f"Brightness baseline accuracy: {df['baseline_correct'].mean():.3f}")
print()
print("By system:")
print(df.groupby('system')[['classifier_correct', 'baseline_correct']].mean())

df.to_csv('/content/field_validation_results.csv', index=False)
print("\nSaved field_validation_results.csv - download from the Colab file browser.")

Field photos (n=31)
Classifier accuracy: 0.677
Brightness baseline accuracy: 0.323

By system:
        classifier_correct  baseline_correct
system                                      
ali               1.000000          0.000000
amir              0.565217          0.434783

Saved field_validation_results.csv - download from the Colab file browser.
